
### DLT PIPELINE: BRONZE LAYER (01_bronze_ingestion)

In [0]:
# -------------------------------------------------------------------------
# Purpose: Ingest raw CSV data using Auto Loader with Schema Evolution & Audit
# Rubric: Domain 3 (DLT, Streaming, Schema Evolution) & Domain 4 (Bronze)
# -------------------------------------------------------------------------

import dlt
from pyspark.sql.functions import current_timestamp, col, lit

# 1. Configuration (Hardcoded for clarity or use spark.conf.get for params)
# The path where your 109M rows are sitting
source_path = "/Volumes/ecommerce_analytics_dev/bronze_layer/raw_landing_volume/raw_landing"

# 2. Define the Bronze Table
@dlt.table(
    name="events_raw",
    comment="Raw ingestion table for e-commerce events with audit trails.",
    table_properties={
        "quality": "bronze",
        "delta.enableChangeDataFeed": "true" # Enables easier tracking for Silver
    }
)
def events_raw():
    return (
        # Use Auto Loader to incrementally ingest CSV files from cloud storage
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "csv")
        # Enable schema inference and evolution to handle new columns automatically
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns") 
        # Capture malformed records in a special column for error handling
        .option("cloudFiles.rescuedDataColumn", "_rescued_data")
        .option("header", "true")
        .load(source_path)
        # Add audit columns: source file path and ingestion timestamps
        .withColumn("source_file", col("_metadata.file_path"))
        .withColumn("ingestion_timestamp", current_timestamp())
        .withColumn("created_at", current_timestamp())
    )